# SC-1-Setup-Foundry - Environnement Smart Contracts

[<< Cypherpunk Origins](SC-0-Cypherpunk-Origins.ipynb) | [Suivant : Setup Web3py >>](SC-2-Setup-Web3py.ipynb)

***

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Installer et configurer **Foundry** (forge, cast, anvil)
2. Verifier l'installation de **Solidity**
3. Créer un premier projet Foundry
4. Compiler et tester un contrat simple

### Prerequis

- Python 3.10+ installe
- curl ou PowerShell disponible
- Git installe

### Duree estimee : 30 minutes


***

## 1. Detection de l'environnement

In [1]:
# Detection de l'environnement
import sys
import platform
import subprocess
from pathlib import Path

OS_NAME = platform.system()
PYTHON_VERSION = sys.version_info

print(f"Systeme: {OS_NAME}")
print(f"Python: {PYTHON_VERSION.major}.{PYTHON_VERSION.minor}.{PYTHON_VERSION.micro}")
print(f"Repertoire de travail: {Path.cwd().name}")

Systeme: Windows
Python: 3.13.12
Repertoire de travail: 00-Foundations


### Interprétation : Détection de l'environnement

**Sortie obtenue** : Affichage des informations système et Python.

| Paramètre | Valeur | Signification |
|-----------|--------|---------------|
| Système | Windows | OS hôte |
| Python | *imprimée par la cellule* — pré-requis : **3.10+** | Version Python installée |
| Répertoire | *imprimé par la cellule* | Nom du répertoire de travail (basename), identique sur toutes les machines |

Ni la version ni le nom du répertoire ne sont recopiés ici : la cellule les
imprime, et sa sortie committée en est la source unique. Ce qui a une valeur
pédagogique est le **pré-requis** `3.10+` — il tient sur toutes les machines,
là où la version installée change d'une machine à l'autre. Pour le répertoire,
la cellule n'imprime que son **basename** (`pathlib.Path.cwd().name`) : le
chemin absolu complet est un artefact de la machine hôte, sans valeur
pédagogique, qui ne doit pas figurer dans les sorties committées (#9434).

**Points clés** :
1. Le pré-requis `3.10+` est satisfait : tous les outils de la série SmartContracts sont compatibles
2. Le basename du répertoire de travail suffit à situer le notebook dans la série ; aucun chemin absolu de la machine hôte n'est imprimé
3. La détection préalable évite les erreurs de compatibilité

> **Note technique** : Si la version Python était < 3.10, certains packages (web3.py v7+) ne fonctionneraient pas.


***

## 2. Installation de Foundry

[Foundry](https://book.getfoundry.sh/) est une suite d'outils moderne pour le développement Ethereum.

### Composants

| Outil | Description |
|-------|-------------|
| **forge** | Compilation, tests, deploiement |
| **cast** | Interactions avec les contrats |
| **anvil** | Noeud Ethereum local |

### Installation

```bash
# Windows (PowerShell)
curl -L https://foundry.paradigm.xyz | bash
foundryup
```

In [2]:
# Verification de Foundry
def check_command(cmd):
    try:
        result = subprocess.run([cmd, '--version'], capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            return True, result.stdout.strip()
        return False, result.stderr
    except FileNotFoundError:
        return False, "Non installe"
    except Exception as e:
        return False, str(e)

print("Verification des outils Foundry...")
print("=" * 50)

for cmd in ['forge', 'cast', 'anvil']:
    ok, info = check_command(cmd)
    status = "OK" if ok else "MANQUANT"
    print(f"{cmd}: {status}")
    if ok:
        print(f"  {info}")

Verification des outils Foundry...
forge: OK
  forge Version: 1.5.1-stable
Commit SHA: b0a9dd9ceda36f63e2326ce530c10e6916f4b8a2
Build Timestamp: 2025-12-22T11:40:33.870895500Z (1766403633)
Build Profile: maxperf
cast: OK
  cast Version: 1.5.1-stable
Commit SHA: b0a9dd9ceda36f63e2326ce530c10e6916f4b8a2
Build Timestamp: 2025-12-22T11:40:33.870895500Z (1766403633)
Build Profile: maxperf
anvil: OK
  anvil Version: 1.5.1-stable
Commit SHA: b0a9dd9ceda36f63e2326ce530c10e6916f4b8a2
Build Timestamp: 2025-12-22T11:40:33.870895500Z (1766403633)
Build Profile: maxperf


### Interprétation : Vérification de l'installation Foundry

**Sortie obtenue** : Les trois outils Foundry sont installés et fonctionnels.

| Outil | Version | Statut |
|-------|---------|--------|
| **forge** | *imprimée par la cellule* | OK |
| **cast** | *imprimée par la cellule* | OK |
| **anvil** | *imprimée par la cellule* | OK |

Les versions ne sont pas recopiées ici : la sortie de la cellule en est la
source unique, et elles changent d'une machine à l'autre (#9434).

**Analyse des composants** :

| Composant | Rôle dans l'écosystème Foundry |
|-----------|-------------------------------|
| **forge** | Gestion de projet, compilation, tests, déploiement |
| **cast** | Interactions avec la blockchain (lecture, écriture) |
| **anvil** | Nœud Ethereum local pour développement et tests |

**Points clés** :
1. Les trois outils partagent la même version (visible dans la sortie) — cohérence garantie
2. Le build profile imprimé par la cellule indique le mode de compilation : `dist` = version de distribution (release), `maxperf` = build optimisé
3. Le commit SHA (visible dans la sortie) permet de reproduire exactement cette version si nécessaire

> **Note technique** : Foundry est écrit en Rust, ce qui le rend beaucoup plus rapide que les outils basés sur JavaScript (comme Truffle ou Hardhat).


***

## 3. Solidity Compiler (solc)

Foundry inclut son propre compilateur Solidity.

In [3]:
# Verification de solc via forge
ok, info = check_command('forge')
if ok:
    print("Solidity est compile via forge build")
    print("Version Foundry:", info)
else:
    print("Foundry non installe - solc non disponible")

Solidity est compile via forge build
Version Foundry: forge Version: 1.5.1-stable
Commit SHA: b0a9dd9ceda36f63e2326ce530c10e6916f4b8a2
Build Timestamp: 2025-12-22T11:40:33.870895500Z (1766403633)
Build Profile: maxperf


### Interprétation : Vérification de Solidity via Foundry

**Sortie obtenue** : Foundry est installé et inclut le compilateur Solidity.

| Composant | Valeur | Signification |
|-----------|--------|---------------|
| Foundry | OK | La suite d'outils est disponible |
| Version | *imprimée par la cellule* | Version Foundry installée sur la machine |

**Points clés** :
1. Foundry inclut son propre compilateur `solc` (pas besoin d'installation séparée)
2. La version Solidity est déterminée par le pragma dans les contrats (`pragma solidity ^0.8.28`)
3. Foundry gère automatiquement les dépendances via `forge-std`

> **Note technique** : Contrairement aux anciens outils (Truffle, Hardhat) qui nécessitaient d'installer `solc` séparément, Foundry embarque tout le nécessaire. Cela simplifie considérablement le setup.


***

## 4. Premier projet Foundry

Creons un projet Foundry de demonstration.

In [4]:
# Creation d'un projet Foundry (demonstration)
import tempfile
import os

# Note: Cette cellule montre les commandes a executer
# En pratique, executez ces commandes dans un terminal

print("Commandes pour creer un projet Foundry:")
print("=" * 50)
print("mkdir my-project && cd my-project")
print("forge init")
print("forge build")
print("forge test")
print("=" * 50)
print("\nCes commandes creent:")
print("  - src/ : Contrats Solidity")
print("  - test/ : Tests Foundry")
print("  - lib/ : Dependances (forge-std)")

Commandes pour creer un projet Foundry:
mkdir my-project && cd my-project
forge init
forge build
forge test

Ces commandes creent:
  - src/ : Contrats Solidity
  - test/ : Tests Foundry
  - lib/ : Dependances (forge-std)


***

## 5. Premier contrat Solidity

Voici un contrat simple pour tester l'installation.

In [5]:
# Affichage du contrat Hello.sol
HELLO_CONTRACT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Hello {
    string public message = "Hello, World!";

    function setMessage(string memory _message) public {
        message = _message;
    }

    function getMessage() public view returns (string memory) {
        return message;
    }
}
'''

print("Contrat Hello.sol:")
print("=" * 50)
print(HELLO_CONTRACT)

Contrat Hello.sol:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Hello {
    string public message = "Hello, World!";

    function setMessage(string memory _message) public {
        message = _message;
    }

    function getMessage() public view returns (string memory) {
        return message;
    }
}



### Interprétation : Structure du contrat Hello.sol

**Sortie obtenue** : Affichage du code source d'un contrat Solidity simple.

| Élément | Ligne | Description |
|---------|-------|-------------|
| **SPDX-License-Identifier** | 1 | Licence MIT (open source) |
| **pragma solidity** | 2 | Version du compilateur requise (0.8.28) |
| **contract Hello** | 4 | Déclaration du contrat |
| **variable public** | 5 | Variable d'état publique avec valeur initiale |
| **setMessage()** | 7-9 | Fonction pour modifier le message |
| **getMessage()** | 11-13 | Fonction pour lire le message |

**Analyse des patterns** :
- **Variable publique** : `string public message` crée automatiquement un getter
- **Fonction de modification** : `setMessage()` permet de changer l'état
- **Fonction de lecture** : `getMessage()` retourne la valeur (redondant avec le getter public)

**Points clés** :
1. Ce contrat illustre les bases de Solidity : variables, fonctions, visibilité
2. Le message est stocké dans le stockage du contrat (coûteux en gas)
3. En production, on utiliserait des events pour les changements d'état

> **Note technique** : Le mot-clé `public` génère automatiquement une fonction getter. Donc `message()` (sans `getMessage()`) suffirait pour lire la valeur.


### Structure du contrat

| Élément | Description |
|---------|-------------|
| `pragma solidity` | Version du compilateur |
| `contract Hello` | Nom du contrat |
| `string public message` | Variable d'etat publique |
| `setMessage()` | Fonction de modification |
| `getMessage()` | Fonction de lecture |

***

## 6. Test Foundry

Les tests Foundry sont ecrits en Solidity.

In [6]:
# Affichage du test Hello.t.sol
TEST_CONTRACT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "forge-std/Test.sol";
import "../src/Hello.sol";

contract HelloTest is Test {
    Hello hello;

    function setUp() public {
        hello = new Hello();
    }

    function testInitialMessage() public view {
        assertEq(hello.message(), "Hello, World!");
    }

    function testSetMessage() public {
        hello.setMessage("Bonjour");
        assertEq(hello.message(), "Bonjour");
    }
}
'''

print("Test Hello.t.sol:")
print("=" * 50)
print(TEST_CONTRACT)

Test Hello.t.sol:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "forge-std/Test.sol";
import "../src/Hello.sol";

contract HelloTest is Test {
    Hello hello;

    function setUp() public {
        hello = new Hello();
    }

    function testInitialMessage() public view {
        assertEq(hello.message(), "Hello, World!");
    }

    function testSetMessage() public {
        hello.setMessage("Bonjour");
        assertEq(hello.message(), "Bonjour");
    }
}



### Interprétation : Structure du test Hello.t.sol

**Sortie obtenue** : Affichage du code source d'un test Foundry.

| Élément | Description |
|---------|-------------|
| **Import Test.sol** | Bibliothèque de test Foundry (`forge-std`) |
| **Import Hello.sol** | Contrat à tester |
| **setUp()** | Initialisation avant chaque test |
| **testInitialMessage()** | Vérifie le message initial |
| **testSetMessage()** | Vérifie la modification du message |

**Patterns de test Foundry** :

| Pattern | Description |
|---------|-------------|
| `contract XTest is Test` | Le contrat de test hérite de `Test` |
| `setUp()` | Exécuté avant chaque test (initialisation) |
| `testXxx()` | Les fonctions commençant par `test` sont exécutées |
| `assertEq(a, b)` | Assertion d'égalité (échoue si `a != b`) |

**Points clés** :
1. Les tests Foundry sont écrits en Solidity (pas de JavaScript/TypeScript)
2. Le pattern `setUp()` + `testXxx()` est standard pour les tests unitaires
3. `assertEq()` est l'assertion la plus courante dans les tests Foundry

> **Note technique** : Pour exécuter ces tests, utilisez `forge test` dans le terminal du projet Foundry. L'option `-vvv` permet d'afficher les traces détaillées en cas d'échec.


***

## 7. Resume de l'environnement

In [7]:
# Resume
print("=" * 60)
print("RESUME DE L'ENVIRONNEMENT SMART CONTRACTS")
print("=" * 60)
print(f"Systeme: {OS_NAME}")
print(f"Python: {PYTHON_VERSION.major}.{PYTHON_VERSION.minor}.{PYTHON_VERSION.micro}")
print("-" * 60)

forge_ok, _ = check_command('forge')
print(f"Foundry: {'OK' if forge_ok else 'MANQUANT (REQUIS)'}")
print("=" * 60)

if forge_ok:
    print("\nEnvironnement pret pour le developpement Smart Contracts.")
else:
    print("\nInstallez Foundry: curl -L https://foundry.paradigm.xyz | bash && foundryup")

RESUME DE L'ENVIRONNEMENT SMART CONTRACTS
Systeme: Windows
Python: 3.13.12
------------------------------------------------------------
Foundry: OK

Environnement pret pour le developpement Smart Contracts.


### Interprétation : Résumé de l'environnement

**Sortie obtenue** : Résumé complet de l'environnement de développement Smart Contracts.

| Composant | Statut | Détails |
|-----------|--------|---------|
| **Système** | Windows | OS compatible avec Foundry |
| **Python** | *imprimé par la cellule* | Pré-requis **3.10+** satisfait |
| **Foundry** | OK | Suite d'outils installée et fonctionnelle |

**État de préparation** :
- ✅ Outils Foundry disponibles (forge, cast, anvil)
- ✅ Python compatible pour les outils web3.py
- ✅ Environnement prêt pour le développement Smart Contracts

**Prochaines étapes** :
1. Installer VSCode et les extensions Solidity (recommandé)
2. Créer un premier projet avec `forge init`
3. Explorer les notebooks suivants de la série

> **Note technique** : Si Foundry n'était pas installé, la commande d'installation Windows serait : `curl -L https://foundry.paradigm.xyz | bash` puis `foundryup`.


***

## 8. VSCode Extensions recommandees

| Extension | Usage |
|-----------|-------|
| **Solidity** (Juan Blanco) | Syntaxe Solidity |
| **Foundry** | Integration Foundry |
| **Error Lens** | Erreurs en ligne |

***

## Points cles

| Concept | Description |
|---------|-------------|
| **Foundry** | Suite d'outils Ethereum moderne |
| **forge** | Compilation et tests |
| **Solidity** | Langage smart contracts |
| **.t.sol** | Convention pour fichiers de test |

***

**Notebook suivant** : [SC-2-Setup-Web3py](SC-2-Setup-Web3py.ipynb)

**Indice : Exercice Premier Projet Foundry**

**Étape 1 - Création du projet**
```bash
# Créez un nouveau répertoire et initialisez le projet
mkdir mon-premier-projet
cd mon-premier-projet
forge init
```
Cette commande crée la structure de base avec `src/`, `test/` et `lib/`.

**Étape 2 - Implémentation du Counter**

**Variables d'état** :
```solidity
uint256 public count;
```

**Fonction `increment()`** :
- Incrémente `count` de 1
- Pas de paramètres nécessaires
- Utilisez `count++` ou `count += 1`

**Fonction `decrement()`** :
- Vérifie que `count > 0` avant de décrémenter
- Utilisez `require(count > 0, "Cannot decrement below zero")`
- Décrémente ensuite

**Fonction `getCount()`** :
- Retourne simplement `count`
- Note : le mot-clé `public` génère déjà un getter automatique

**Étape 3 - Tests**

**Structure du test** :
```solidity
contract CounterTest is Test {
    Counter counter;
    
    function setUp() public {
        counter = new Counter();
    }
    
    function testIncrement() public {
        counter.increment();
        assertEq(counter.getCount(), 1);
    }
    
    function testDecrement() public {
        counter.increment();
        counter.increment();
        counter.decrement();
        assertEq(counter.getCount(), 1);
    }
    
    function testDecrementZero() public {
        // Ce test doit vérifier que decrement() échoue si count == 0
        // Indice : utilisez vm.expectRevert() avant l'appel
    }
}
```

**Étape 4 - Exécution**
```bash
forge build
forge test -vvv
```

> **Note** : Les fonctions `public` génèrent automatiquement des getters. `counter.count()` équivaut à `counter.getCount()`.


## Resume et perspectives

Ce notebook a guide l'installation et la verification de la suite Foundry (forge, cast, anvil), l'environnement de développement moderne pour les smart contracts Ethereum. Les trois composants couvrent l'ensemble du cycle de vie : `forge` pour la compilation, les tests et le deploiement des contrats Solidity ; `cast` pour les interactions avec la blockchain depuis la ligne de commande ; et `anvil` comme noeud Ethereum local fournissant 10 comptes pre-finances pour le développement et les tests. L'ecriture de tests en Solidity natif (plutot qu'en JavaScript) est l'un des avantages majeurs de Foundry par rapport aux alternatives comme Hardhat ou Truffle.

Le contrat `Hello.sol` et son test `Hello.t.sol` ont illustre les conventions fondamentales de Foundry : le pattern `setUp()` + `testXxx()` pour les tests unitaires, les assertions `assertEq()` pour la verification, et la structure de projet `src/` / `test/` / `lib/`. Le mot-cle `public` generant automatiquement un getter est un premier apercu de la magie Solidity qui sera detaillee dans les notebooks suivants. L'exercice du contrat Counter avec `increment()`, `decrement()` et `vm.expectRevert()` prepare aux patterns de test avance.

L'environnement etant operationnel, le notebook suivant [SC-2-Setup-Web3py](SC-2-Setup-Web3py.ipynb) complete la chaîne d'outils en ajoutant l'interaction Python avec Ethereum via web3.py, permettant de compiler, deployer et appeler des contrats Solidity entierement depuis un notebook Jupyter.

***

[<< Cypherpunk Origins](SC-0-Cypherpunk-Origins.ipynb) | [Suivant : Setup Web3py >>](SC-2-Setup-Web3py.ipynb)
